# 🎯 PRESENTASI DATA SCIENCE - PERTEMUAN 11
## **Unsupervised Learning: Clustering (K-Means & Hierarchical)**

---

**Nama:** [Isi Nama Kamu]  
**NIM:** [Isi NIM Kamu]  
**Kelas:** [Isi Kelas Kamu]

---

### 📌 Alasan Memilih Topik Ini:
Clustering adalah teknik **Unsupervised Learning** yang paling fundamental dan banyak digunakan di industri. Teknik ini memiliki:
- ✅ **Visualisasi yang menarik** (scatter plot berwarna, dendrogram)
- ✅ **Aplikasi nyata** yang luas (segmentasi pelanggan, pengelompokan data)
- ✅ **Konsep intuitif** yang mudah dipahami dan dijelaskan
- ✅ **Implementasi sederhana** dengan scikit-learn

---
## 📚 OUTLINE PRESENTASI

| No | Topik | Durasi |
|----|-------|--------|
| 1 | Pengantar Clustering & Unsupervised Learning | 2 menit |
| 2 | Eksplorasi Dataset (EDA) | 3 menit |
| 3 | Preprocessing Data | 2 menit |
| 4 | K-Means Clustering - Teori & Implementasi | 5 menit |
| 5 | Metode Elbow untuk Menentuan K Optimal | 3 menit |
| 6 | Visualisasi & Interpretasi Hasil | 3 menit |
| 7 | Hierarchical Clustering & Dendrogram | 3 menit |
| 8 | Kesimpulan | 2 menit |

## 🔧 BAGIAN 1: SETUP & IMPORT LIBRARY

**Penjelasan untuk Presentasi:**
- Kita menggunakan library utama dalam Data Science Python:
  - `numpy` & `pandas` → manipulasi data
  - `matplotlib` & `seaborn` → visualisasi
  - `sklearn` → machine learning (clustering)
  - `scipy` → hierarchical clustering

In [ ]:
# ============================================
# IMPORT LIBRARY UTAMA
# ============================================

# Library untuk manipulasi data
import numpy as np
import pandas as pd

# Library untuk visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Setting agar visualisasi lebih baik
plt.rcParams['figure.figsize'] = 10, 6
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Library untuk Machine Learning (Clustering)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs

# Library untuk Hierarchical Clustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Import warnings supaya output bersih
import warnings
warnings.filterwarnings('ignore')

print("✅ Semua library berhasil di-import!")
print("\n📦 Library yang digunakan:")
print(f"   - NumPy versi: {np.__version__}")
print(f"   - Pandas versi: {pd.__version__}")

---
## 📊 BAGIAN 2: PEMBUATAN & EKSPLORASI DATASET

### 🎯 Konteks Bisnis: **Segmentasi Pelanggan**

**Cerita untuk Presentasi:**
> "Bayangkan kita adalah Data Scientist di sebuah perusahaan retail.  
> Kita memiliki data **200 pelanggan** dengan informasi:
> - **Pendapatan Tahunan** (dalam juta Rupiah)
> - **Skor Belanja** (1-100, semakin tinggi = semakin boros)
>  
> **Tugas kita:** Kelompokkan pelanggan menjadi segmen yang mirip, tanpa label sebelumnya!"

In [ ]:
# ============================================
# GENERATE DATASET SYNTHETIC (DATA PELANGGAN)
# ============================================

# Set random seed agar hasil reproducible
np.random.seed(42)

# Buat 3 kelompok pelanggan yang tersembunyi (ground truth)
# Kelompok 1: Pelanggan HEMAT (pendapatan rendah, belanja rendah)
grp1 = np.random.normal(loc=[30, 20], scale=[6, 8], size=(100, 2))

# Kelompok 2: Pelanggan MENENGAH (pendapatan menengah, belanja menengah)
grp2 = np.random.normal(loc=[70, 55], scale=[8, 10], size=(100, 2))

# Kelompok 3: Pelanggan PREMIUM/BOROS (pendapatan tinggi, belanja tinggi)
grp3 = np.random.normal(loc=[110, 85], scale=[10, 8], size=(100, 2))

# Gabungkan semua kelompok
data = np.vstack([grp1, grp2, grp3])

# Buat DataFrame
df = pd.DataFrame(data, columns=['Pendapatan_Tahunan', 'Skor_Belanja'])

# Tambah kolom tambahan (opsional untuk analisis)
df['Usia'] = np.random.randint(18, 65, len(df))
df['Gender'] = np.random.choice(['L', 'P'], len(df))

print(f"✅ Dataset berhasil dibuat!")
print(f"📊 Jumlah pelanggan: {len(df)}")
print(f"📐 Jumlah fitur: {df.shape[1]}")

In [ ]:
# ============================================
# EKSPLORASI DATA (EXPLORATORY DATA ANALYSIS)
# ============================================

print("=" * 50)
print("📈 STATISTIKA DESKRIPTIF")
print("=" * 50)
display(df.describe().round(2))

print("\n" + "=" * 50)
print("🔍 5 BARIS PERTAMA DATA")
print("=" * 50)
display(df.head())

print("\n" + "=" * 50)
print("❓ INFO DATA (Missing Values & Tipe Data)")
print("=" * 50)
display(df.info())

In [ ]:
# ============================================
# VISUALISASI SEBARAN DATA AWAL
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scatter Plot Pendapatan vs Skor Belanja
axes[0].scatter(df['Pendapatan_Tahunan'], df['Skor_Belanja'], 
               alpha=0.6, c='#3498db', edgecolors='white', s=50)
axes[0].set_xlabel('Pendapatan Tahunan (juta Rp)', fontsize=12)
axes[0].set_ylabel('Skor Belanja (1-100)', fontsize=12)
axes[0].set_title('📊 Sebaran Data Pelanggan\n(Sebelum Clustering)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: Distribusi masing-masing fitur
axes[1].hist(df['Pendapatan_Tahunan'], bins=20, alpha=0.7, label='Pendapatan', color='#2ecc71', edgecolor='white')
axes[1].hist(df['Skor_Belanja'], bins=20, alpha=0.7, label='Skor Belanja', color='#e74c3c', edgecolor='white')
axes[1].set_xlabel('Nilai', fontsize=12)
axes[1].set_ylabel('Frekuensi', fontsize=12)
axes[1].set_title('📈 Distribusi Fitur', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 OBSERVASI dari visualisasi:")
print("   • Terlihat ada ~3 kelompok data yang terpisah secara alami")
print("   • Kelompok kiri bawah: pendapatan & belanja RENDAH")
print("   • Kelompok tengah: pendapatan & belanja MENENGAH")
print("   • Kelompok kanan atas: pendapatan & belanja TINGGI")

---
## ⚙️ BAGIAN 3: PREPROCESSING DATA

### ❓ Kenapa Harus Scaling?

**Penjelasan untuk Presentasi:**
> "K-Means menggunakan **jarak Euclidean** untuk mengukur kemiripan.  
> Jika satu fitur bernilai besar (misal: pendapatan 30-150 juta) dan lainnya kecil (skor 1-100),  
> maka fitur dengan nilai besar akan **mendominasi** perhitungan jarak!"

**Solusi:** Gunakan **StandardScaler** → membuat semua fitur memiliki:
- Mean = 0
- Standard Deviation = 1

In [ ]:
# ============================================
# PREPROCESSING: STANDARD SCALER
# ============================================

# Pilih fitur yang akan digunakan untuk clustering
fitur_clustering = ['Pendapatan_Tahunan', 'Skor_Belanja']
X = df[fitur_clustering].values

# Buat objek StandardScaler
scaler = StandardScaler()

# Fit dan transform data
X_scaled = scaler.fit_transform(X)

# Buat DataFrame dari data yang sudah di-scale
df_scaled = pd.DataFrame(X_scaled, columns=['Pendapatan_Scaled', 'Skor_Scaled'])

print("✅ Data berhasil di-scale menggunakan StandardScaler!")
print("\n📊 STATISTIK SETELAH SCALING:")
print(f"   - Mean Pendapatan: {X_scaled[:, 0].mean():.6f} (harusnya ≈ 0)")
print(f"   - Std  Pendapatan: {X_scaled[:, 0].std():.6f} (harusnya ≈ 1)")
print(f"   - Mean Skor Belanja: {X_scaled[:, 1].mean():.6f} (harusnya ≈ 0)")
print(f"   - Std  Skor Belanja: {X_scaled[:, 1].std():.6f} (harusnya ≈ 1)")

In [ ]:
# Visualisasi perbandingan SEBELUM dan SESUDAH scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sebelum Scaling
axes[0].scatter(X[:, 0], X[:, 1], alpha=0.6, c='#3498db', edgecolors='white', s=50)
axes[0].set_xlabel('Pendapatan Tahunan (juta Rp)', fontsize=11)
axes[0].set_ylabel('Skor Belanja (1-100)', fontsize=11)
axes[0].set_title('❌ SEBELUM Scaling\n(Skala berbeda)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Sesudah Scaling
axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], alpha=0.6, c='#27ae60', edgecolors='white', s=50)
axes[1].xlabel = 'Pendapatan (Scaled)'
axes[1].ylabel = 'Skor Belanja (Scaled)'
axes[1].set_title('✅ SESUDAH StandardScaler\n(Skala seragam)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

print("💡 Perhatikan bagaimana skala sumbu sekarang SERAGAM!")

---
## 🎯 BAGIAN 4: K-MEANS CLUSTERING - TEORI

### Apa itu K-Means?

**K-Means** adalah algoritma clustering **partitional** yang paling populer. Caranya:

1. **Tentukan K** (jumlah cluster yang diinginkan)
2. **Inisialisasi** K centroid (titik pusat) secara acak
3. **Assignment**: Setiap titik data → centroid terdekat
4. **Update**: Centroid baru = rata-rata anggota cluster
5. **Ulangi** langkah 3-4 hingga **konvergen**

---

### 📐 Fungsi Objektif: WCSS (Within-Cluster Sum of Squares)

$$WCSS = \sum_{k} \sum_{x \in C_k} ||x - \mu_k||^2$$

Dimana:
- **K** = jumlah cluster
- **μₖ** = centroid cluster ke-k
- **x** = titik data dalam cluster

**Tujuan:** Minimalkan WCSS → cluster semakin **kompak/rapat**

### 🔄 Visualisasi Algoritma K-Means Step-by-Step

```
ITERASI 1:                    ITERASI 2:                    ITERASI 3 (KONVERGEN):
    ●                       ●                           ●
   /|\                     /|\                         /|\
  / | \        →          / | \         →             / | \
 ●  ●  ●               ●  X  ●                     ●  X  ●
  \ | /                   \ | /                       \ | /
   \|/                     \|/                         \│/
    ●                       ●                           ●

 Centroid acak           Update posisi              Centroid stabil!
 Assignment pertama       (rata-rata anggota)         (tidak berubah)
```

---
## 📈 BAGIAN 5: METODE ELBOW - MENENTUKAN K OPTIMAL

### ❓ Berapa K yang Tepat?

**Masalah utama K-Means:** Kita harus tentukan **K terlebih dahulu**!

**Solusi: Metode Elbow**
- Plot nilai **WCSS** untuk berbagai K (biasanya 1-10)
- Cari titik **"siku" (elbow)** pada kurva
- Titik elbow = K optimal

---

### 📊 Cara Maca Grafik Elbow:

| Posisi Kurva | Interpretasi |
|--------------|-------------|
| **Sebelum Elbow** | Penurunan WCSS masih tajam → K belum cukup |
| **Titik Elbow** | ⭐ **K optimal** - keseimbangan terbaik |
| **Setelah Elbow** | Penurunan kecil → waspadai overfitting |

In [ ]:
# ============================================
# METODE ELBOW: CARI K OPTIMAL
# ============================================

# Simpan nilai WCSS untuk setiap K
wcss_values = []
K_range = range(1, 11)  # Coba K dari 1 sampai 10

# Hitung WCSS untuk setiap nilai K
for k in K_range:
    # Buat model K-Means dengan K clusters
    kmeans_temp = KMeans(n_clusters=k, 
                          init='k-means++',  # Inisialisasi cerdas (default)
                          random_state=42,
                          n_init=10)
    kmeans_temp.fit(X_scaled)
    wcss_values.append(kmeans_temp.inertia_)  # inertia_ = WCSS

# Plot Grafik Elbow
plt.figure(figsize=(10, 6))
plt.plot(K_range, wcss_values, marker='o', linewidth=2, 
         markersize=8, color='#3498db', markerfacecolor='red')

# Tandai titik elbow (biasanya di K=3 untuk data ini)
plt.axvline(x=3, color='red', linestyle='--', linewidth=2, label='K=3 (Elbow)')

# Anotasi
plt.annotate('📍 TITIK ELBOW\nK = 3', 
             xy=(3, wcss_values[2]), 
             xytext=(4.5, wcss_values[2]+50),
             fontsize=11,
             arrowprops=dict(arrowstyle='->', color='red'),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.xlabel('Jumlah Cluster (K)', fontsize=12, fontweight='bold')
plt.ylabel('WCSS (Inertia)', fontsize=12, fontweight='bold')
plt.title('📊 GRAFIK METODE ELBOW\nCari K Optimal untuk Segmentasi Pelanggan', 
          fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Tampilkan nilai WCSS di setiap titik
for i, (k, wcss) in enumerate(zip(K_range, wcss_values)):
    if k <= 6:  # Hanya tampilkan untuk K kecil agar tidak crowded
        plt.annotate(f'{wcss:.0f}', (k, wcss), textcoords="offset points", 
                    xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("📋 TABEL NILAI WCSS PER K:")
print("="*60)
elbow_df = pd.DataFrame({
    'K (Jumlah Cluster)': list(K_range),
    'WCSS': [f'{w:.2f}' for w in wcss_values],
    'Penurunan (Δ)': ['-'] + [f'{wcss_values[i-1]-wcss_values[i]:.2f}' for i in range(1, len(wcss_values))]
})
display(elbow_df)

print("\n🎯 INTERPRETASI:")
print("   • Dari K=1 ke K=2: Penurunan BESAR (masih bisa diperbaiki)")
print("   • Dari K=2 ke K=3: Penurunan MASIH SIGNIFIKAN ⭐")
print("   • Dari K=3 ke K=4+: Penurunan MELANDAI → ELBOW DI K=3!")

---
## 🏆 BAGIAN 6: IMPLEMENTASI K-MEANS DENGAN K OPTIMAL

### Sekarang kita latih model K-Means dengan **K = 3**!

In [ ]:
# ============================================
# TRAIN MODEL K-MEANS DENGAN K=3
# ============================================

# Set parameter optimal
K_OPTIMAL = 3

# Buat dan train model K-Means
model_kmeans = KMeans(
    n_clusters=K_OPTIMAL,      # Jumlah cluster (dari Elbow Method)
    init='k-means++',          # Inisialisasi cerdas (lebih stabil dari random)
    random_state=42,           # Reproducible
    n_init=10,                 # Jumlah kali running dengan init berbeda
    max_iter=300               # Maksimum iterasi
)

# Fit model ke data
model_kmeans.fit(X_scaled)

# Prediksi label cluster
labels = model_kmeans.labels_

# Dapatkan koordinat centroid (dalam scaled space)
centroids_scaled = model_kmeans.cluster_centers_

# Transform centroid kembali ke original scale
centroids_original = scaler.inverse_transform(centroids_scaled)

# Hitung metrik evaluasi
wcss_final = model_kmeans.inertia_
silhouette_avg = silhouette_score(X_scaled, labels)

print("✅ Model K-Means BERHASIL dilatih!")
print("\n" + "="*50)
print("📊 HASIL TRAINING MODEL:")
print("="*50)
print(f"   ✓ Jumlah Cluster (K):     {K_OPTIMAL}")
print(f"   ✓ WCSS Akhir:            {wcss_final:.3f}")
print(f"   ✓ Silhouette Score:      {silhouette_avg:.3f}")
print(f"   ✓ Jumlah Iterasi:        {model_kmeans.n_iter_}")

print("\n📍 KOORDINAT CENTROID (dalam skala asli):")
centroid_df = pd.DataFrame(
    centroids_original, 
    columns=['Pendapatan_Tahunan', 'Skor_Belanja']
)
centroid_df.index.name = 'Cluster'
display(centroid_df.round(2))

In [ ]:
# Tambahkan label cluster ke DataFrame asli
df['Cluster'] = labels

# Analisis karakteristik setiap cluster
print("="*60)
print("📊 KARAKTERISTIK SETIAP CLUSTER:")
print("="*60)

cluster_analysis = df.groupby('Cluster')[['Pendapatan_Tahunan', 'Skor_Belanja']].agg(['mean', 'count'])
cluster_analysis.columns = ['Rata_Pendapatan', 'Jumlah', 'Rata_Skor', '_']
cluster_analysis = cluster_analysis[['Rata_Pendapatan', 'Rata_Skor', 'Jumlah']]
cluster_analysis['Persentase (%)'] = (cluster_analysis['Jumlah'] / len(df) * 100).round(1)

display(cluster_analysis.round(2))

# Beri nama segment
segment_names = {
    0: '💰 SEGMENT HEMAT' if cluster_analysis.loc[0, 'Rata_Pendapatan'] < 50 else '💰 SEGMENT PREMIUM',
    1: '🛒 SEGMENT MENENGAH',
    2: '👑 SEGMENT PREMIUM' if cluster_analysis.loc[2, 'Rata_Pendapatan'] > 80 else '💵 SEGMENT HEMAT'
}

print("\n🏷️ NAMA SEGMENT (berdasarkan karakteristik):")
for idx, row in cluster_analysis.iterrows():
    if row['Rata_Pendapatan'] < 50:
        nama = "💰 SEGMENT HEMAT (Budget Conscious)"
    elif row['Rata_Pendapatan'] < 90:
        nama = "🛒 SEGMENT MENENGAH (Standard)"
    else:
        nama = "👑 SEGMENT PREMIUM (High Value)"
    print(f"   Cluster {idx}: {nama}")
    print(f"      → Rata-rata Pendapatan: Rp {row['Rata_Pendapatan']:.1f} juta/tahun")
    print(f"      → Rata-rata Skor Belanja: {row['Rata_Skor']:.1f}/100")
    print(f"      → Jumlah Pelanggan: {int(row['Jumlah'])} orang ({row['Persentase (%)']}%)")
    print()

---
## 🎨 BAGIAN 7: VISUALISASI HASIL CLUSTERING

### Ini adalah bagian PALING PENTING untuk presentasi!
**Visualisasi yang menarik akan membantu audiens memahami hasil clustering.**

In [ ]:
# ============================================
# VISUALISASI HASIL K-MEANS CLUSTERING
# ============================================

# Palet warna untuk cluster
colors = ['#3498db', '#e74c3c', '#2ecc71']  # Biru, Merah, Hijau
markers = ['o', 's', '^']  # Lingkaran, Kotak, Segitiga

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ========== PLOT 1: HASIL CLUSTERING ==========
for cluster_id in range(K_OPTIMAL):
    # Filter data untuk cluster ini
    mask = labels == cluster_id
    axes[0].scatter(df.loc[mask, 'Pendapatan_Tahunan'], 
                   df.loc[mask, 'Skor_Belanja'],
                   c=colors[cluster_id], 
                   marker=markers[cluster_id],
                   s=80, 
                   alpha=0.7, 
                   label=f'Cluster {cluster_id}',
                   edgecolors='white',
                   linewidths=0.5)

# Plot centroid
axes[0].scatter(centroids_original[:, 0], 
               centroids_original[:, 1],
               c='black', 
               marker='X', 
               s=300, 
               label='CENTROID',
               edgecolors='gold',
               linewidths=2,
               zorder=5)

axes[0].set_xlabel('Pendapatan Tahunan (juta Rp)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Skor Belanja (1-100)', fontsize=12, fontweight='bold')
axes[0].set_title('🎯 HASIL SEGMENTASI PELANGGAN\n(K-Means Clustering, K=3)', 
                  fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10, loc='upper left')
axes[0].grid(True, alpha=0.3)

# ========== PLOT 2: DENGAN ANOTASI SEGMENT ==========
for cluster_id in range(K_OPTIMAL):
    mask = labels == cluster_id
    axes[1].scatter(df.loc[mask, 'Pendapatan_Tahunan'], 
                   df.loc[mask, 'Skor_Belanja'],
                   c=colors[cluster_id], 
                   marker=markers[cluster_id],
                   s=80, 
                   alpha=0.7, 
                   edgecolors='white',
                   linewidths=0.5)

# Plot centroid
axes[1].scatter(centroids_original[:, 0], 
               centroids_original[:, 1],
               c='black', 
               marker='X', 
               s=300,
               edgecolors='gold',
               linewidths=2,
               zorder=5)

# Tambahkan anotasi nama segment
axes[1].annotate('💰 HEMAT\n(Pendapatan Rendah)', 
                xy=(centroids_original[0, 0], centroids_original[0, 1]),
                xytext=(centroids_original[0, 0]-25, centroids_original[0, 1]+15),
                fontsize=10, ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#3498db', alpha=0.3),
                arrowprops=dict(arrowstyle='->', color='#3498db'))

axes[1].annotate('🛒 MENENGAH\n(Standar)', 
                xy=(centroids_original[1, 0], centroids_original[1, 1]),
                xytext=(centroids_original[1, 0]+25, centroids_original[1, 1]-10),
                fontsize=10, ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#e74c3c', alpha=0.3),
                arrowprops=dict(arrowstyle='->', color='#e74c3c'))

axes[1].annotate('👑 PREMIUM\n(High Value)', 
                xy=(centroids_original[2, 0], centroids_original[2, 1]),
                xytext=(centroids_original[2, 0]-20, centroids_original[2, 1]+15),
                fontsize=10, ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#2ecc71', alpha=0.3),
                arrowprops=dict(arrowstyle='->', color='#2ecc71'))

axes[1].set_xlabel('Pendapatan Tahunan (juta Rp)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Skor Belanja (1-100)', fontsize=12, fontweight='bold')
axes[1].set_title('📝 INTERPRETASI SEGMENT\n(Strategi Pemasaran)', 
                  fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("🎯 REKOMENDASI STRATEGI PEMASARAN PER SEGMENT:")
print("="*70)
print("""
┌─────────────┬─────────────────────────────────────────────────────────┐
│  SEGMENT    │  STRATEGI PEMASARAN                                  │
├─────────────┼─────────────────────────────────────────────────────────┤
│ 💰 HEMAT    │  • Fokus diskon & promo bundling                    │
│             │  • Produk value-pack / ekonomis                    │
│             │  • Komunikasi via WhatsApp / media sosial gratis     │
├─────────────┼─────────────────────────────────────────────────────────┤
│ 🛒 MENENGAH  │  • Program loyalitas & cashback                    │
│             │  • Cross-sell produk mid-range                      │
│             │  • Email marketing & newsletter                    │
├─────────────┼─────────────────────────────────────────────────────────┤
│ 👑 PREMIUM   │  • VIP membership & exclusive deals                 │
│             │  • Personal shopping assistant                     │
│             │  • Produk premium & limited edition                 │
└─────────────┴─────────────────────────────────────────────────────────┘
""")

---
## 📊 SILHOUETTE SCORE - VALIDASI CLUSTER

### Apa itu Silhouette Score?

**Silhouette Score** mengukur **seberapa baik** suatu titik data cocok dengan cluster-nya sendiri dibanding cluster tetangga.

**Rentang Nilai:**
- **Mendekati +1** → Cluster sangat baik (terpisah jauh)
- **Mendekati 0** → Titik di batas antara 2 cluster
- **Mendekati -1** → Titik salah kelompok

---

### Rumus Silhouette:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

Dimana:
- **a(i)** = rata-rata jarak ke titik lain dalam cluster yang sama
- **b(i)** = rata-rata jarak ke titik di cluster terdekat lainnya

In [ ]:
# ============================================
# HITUNG SILHOUETTE SCORE UNTUK BERBAGAI K
# ============================================

silhouette_scores = []
K_range_sil = range(2, 11)  # Minimal 2 cluster untuk silhouette

for k in K_range_sil:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(X_scaled)
    sil_temp = silhouette_score(X_scaled, km_temp.labels_)
    silhouette_scores.append(sil_temp)
    print(f"   K={k}: Silhouette Score = {sil_temp:.3f}")

# Plot Silhouette Scores
fig, ax1 = plt.subplots(figsize=(12, 6))

# Bar plot untuk silhouette
bars = ax1.bar(K_range_sil, silhouette_scores, color='#9b59b6', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Jumlah Cluster (K)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Silhouette Score', fontsize=12, fontweight='bold', color='#9b59b6')
ax1.tick_params(axis='y', labelcolor='#9b59b6')
ax1.set_ylim(0, 1)
ax1.axhline(y=silhouette_avg, color='red', linestyle='--', linewidth=2, label=f'Silhouette K={K_OPTIMAL}: {silhouette_avg:.3f}')

# Tambahkan nilai di atas bar
for bar, score in zip(bars, silhouette_scores):
    height = bar.get_height()
    ax1.annotate(f'{score:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Highlight bar untuk K optimal
bars[K_OPTIMAL-2].set_color('#e74c3c')
bars[K_OPTIMAL-2].set_edgecolor('gold')
bars[K_OPTIMAL-2].set_linewidth(3)

plt.title('📊 SILHOUETTE SCORE UNTUK BERBAGAI NILAI K\n(Validasi Kualitas Cluster)', 
          fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n🎯 INTERPRETASI SILHOUETTE SCORE:")
if silhouette_avg > 0.5:
    print(f"   ✅ Silhouette Score = {silhouette_avg:.3f} → Cluster TERPISAH BAIK!")
elif silhouette_avg > 0.25:
    print(f"   ⚠️ Silhouette Score = {silhouette_avg:.3f} → Structure cukup jelas")
else:
    print(f"   ❌ Silhouette Score = {silhouette_avg:.3f} → Perlu evaluasi ulang")

---
## 🌳 BAGIAN 8: HIERARCHICAL CLUSTERING

### Apa itu Hierarchical Clustering?

**Hierarchical Clustering** membangun **hierarki cluster** (seperti pohon) tanpa perlu tentukan K di awal.

**Dua Pendekatan:**
1. **Agglomerative (Bottom-Up)**: Mulai dari tiap titik sebagai cluster sendiri, lalu GABUNG bertahap → **PALING UMUM**
2. **Divisive (Top-Down)**: Mulai dari 1 cluster besar, lalu PECAH bertahap

---

### Metode Linkage (Ukuran Jarak Antar Cluster):

| Metode | Definisi | Karakteristik |
|--------|----------|---------------|
| **Single** | Jarak minimum | Rentan chaining |
| **Complete** | Jarak maksimum | Cluster kompak |
| **Average** | Rata-rata semua pasangan | Kompromi |
| **Ward** | Minimalkan variansi | ⭐ **Paling umum** |

In [ ]:
# ============================================
# HIERARCHICAL CLUSTERING - DENDROGRAM
# ============================================

# Hitung linkage matrix menggunakan Ward method
Z = linkage(X_scaled, method='ward')

# Plot Dendrogram
plt.figure(figsize=(14, 7))

dendrogram(
    Z,
    leaf_rotation=90,           # Rotasi label 90 derajat
    leaf_font_size=10,          # Ukuran font label
    show_contracted=True,       # Tampilkan kontraksi untuk data banyak
    truncate_mode='lastp',       # Tampilkan p cluster terakhir
    p=30,                        # Jumlah cluster yang ditampilkan
    show_leaf_counts=True,      # Tampilkan jumlah anggota
    above_threshold_color='#e74c3c',
    color_threshold=15          # Threshold untuk warna
)

# Garis potong untuk menentukan K (contoh: K=3)
plt.axhline(y=15, color='red', linestyle='--', linewidth=2, label='Garis Potong (K=3)')

plt.title('🌳 DENDROGRAM - HIERARCHICAL CLUSTERING\n(Ward Linkage Method)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Indeks Data atau (Ukuran Cluster)', fontsize=12, fontweight='bold')
plt.ylabel('Jarak (Ward)', fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("📖 CARA MEMACA DENDROGRAM:")
print("="*60)
print("""
1. Setiap daun (bottom) = 1 data point
2. Tinggi vertikal = JARAK saat 2 cluster digabung
3. Semakin tinggi gabungan → semakin beda 2 cluster tersebut
4. Untuk dapat K cluster: POTONG dendogram secara horizontal!
5. Jumlah garis vertikal yang terpotong = jumlah cluster

🔍 PADA DENDROGRAM DI ATAS:
   • Garis merah putus-putus = potongan untuk K=3
   • Terlihat 3 cabang utama dengan warna berbeda
   • Hasilnya KONSISTEN dengan K-Means (K=3)! ✓
""")

In [ ]:
# ============================================
# PERBANDINGAN: K-MEANS VS HIERARCHICAL
# ============================================

from scipy.cluster.hierarchy import fcluster

# Potong dendrogram untuk dapat 3 cluster
hierarchical_labels = fcluster(Z, t=3, criterion='maxclust')

# Buat figure perbandingan
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ========== PLOT 1: K-MEANS ==========
for i in range(3):
    mask = labels == i
    axes[0].scatter(df.loc[mask, 'Pendapatan_Tahunan'], 
                   df.loc[mask, 'Skor_Belanja'],
                   c=colors[i], marker=markers[i], s=60, alpha=0.7, label=f'Cluster {i}')
axes[0].scatter(centroids_original[:, 0], centroids_original[:, 1],
               c='black', marker='X', s=200, edgecolors='gold', linewidths=2)
axes[0].set_title('K-MEANS\n(K=3)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Pendapatan Tahunan')
axes[0].set_ylabel('Skor Belanja')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ========== PLOT 2: HIERARCHICAL ==========
for i in range(1, 4):
    mask = hierarchical_labels == i
    axes[1].scatter(df.loc[mask, 'Pendapatan_Tahunan'], 
                   df.loc[mask, 'Skor_Belanja'],
                   c=colors[i-1], marker=markers[i-1], s=60, alpha=0.7, label=f'Cluster {i}')
axes[1].set_title('HIERARCHICAL\n(Ward, K=3)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pendapatan Tahunan')
axes[1].set_ylabel('Skor Belanja')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ========== PLOT 3: GROUND TRUTH ==========
# Buat ground truth dari data sintetis
ground_truth = np.array([0]*100 + [1]*100 + [2]*100)
for i in range(3):
    mask = ground_truth == i
    axes[2].scatter(df.iloc[mask]['Pendapatan_Tahunan'], 
                   df.iloc[mask]['Skor_Belanja'],
                   c=colors[i], marker=markers[i], s=60, alpha=0.7, label=f'True Group {i}')
axes[2].set_title('GROUND TRUTH\n(Data Asli)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Pendapatan Tahunan')
axes[2].set_ylabel('Skor Belanja')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('⚖️ PERBANDINGAN: K-MEANS vs HIERARCHICAL vs GROUND TRUTH', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Hitung kesesuaian
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari_kmeans = adjusted_rand_score(ground_truth, labels)
ari_hier = adjusted_rand_score(ground_truth, hierarchical_labels)

print("\n" + "="*60)
print("⚖️ EVALUASI PERBANDINGAN:")
print("="*60)
print(f"\n   📊 Adjusted Rand Index (vs Ground Truth):")
print(f"      • K-Means:      {ari_kmeans:.3f}")
print(f"      • Hierarchical: {ari_hier:.3f}")
print(f"\n   💡 Interpretasi:")
print(f"      • ARI mendekati 1 = clustering SANGAT SESUAI ground truth")
print(f"      • Kedua algoritma menghasilkan hasil yang SANGAT SIMILAR!")

---
## 📋 RINGKASAN & KESIMPULAN

### ✅ Apa yang Sudah Kita Pelajari:

| Topik | Poin Penting |
|-------|-------------|
| **Unsupervised Learning** | Machine learning tanpa label, cari pola tersembunyi |
| **Clustering** | Mengelompokkan data berdasarkan kemiripan |
| **K-Means** | Algoritma partitional, butuh K, cepat & skalabel |
| **WCSS** | Fungsi objektif: total jarak kuadrat ke centroid |
| **Metode Elbow** | Cara visual menentukan K optimal |
| **Silhouette Score** | Validasi kualitas cluster (-1 sampai +1) |
| **Hierarchical** | Clustering bertingkat, hasilkan dendrogram |
| **Dendrogram** | Visualisasi hierarki cluster |

---

### 🎯 Kesimpulan Akhir:

1. **K-Means** cocok untuk dataset besar, cluster berbentuk bulat, K sudah diketahui
2. **Metode Elbow** membantu menentukan K optimal secara visual
3. **Hierarchical Clustering** memberikan insight tambahan melalui dendrogram
4. **Kedua algoritma** menghasilkan hasil yang konsisten untuk data ini
5. **Segmentasi pelanggan** dapat langsung diaplikasikan untuk strategi pemasaran

---

### 🚀 Aplikasi di Dunia Nyata:
- 🛒 **E-commerce**: Segmentasi pelanggan, rekomendasi produk
- 🏥 **Kesehatan**: Segmentasi pasien berdasarkan gejala
- 📱 **Marketing**: Targeted advertising, personalisasi konten
- 🏦 **Perbankan**: Credit scoring, fraud detection
- 🎬 **Entertainment**: Rekomendasi film/music (Spotify, Netflix)

---
## 🙏 TERIMA KASIH

### Pertanyaan & Diskusi?

---

**Referensi:**
- Géron, A. (2022). *Hands-On Machine Learning* - Ch 9
- scikit-learn Documentation: https://scikit-learn.org/
- Modul Pertemuan 11 - Data Science

---

*Notebook ini dibuat untuk keperluan presentasi mata kuliah Data Science*

In [ ]:
# ============================================
# BONUS: SAVE RESULTS TO FILE
# ============================================

# Simpan hasil clustering ke CSV
output_file = 'hasil_segmentasi_pelanggan.csv'  # disimpan di folder kerja notebook (Colab: /content)
df.to_csv(output_file, index=False)

print(f"✅ Hasil clustering disimpan ke: {output_file}")
print(f"\n📄 Preview file:")
display(df.head(10))

# Summary statistik per cluster
print("\n" + "="*60)
print("📊 SUMMARY AKHIR:")
print("="*60)
summary = df.groupby('Cluster').agg({
    'Pendapatan_Tahunan': ['count', 'mean', 'min', 'max'],
    'Skor_Belanja': ['mean', 'min', 'max'],
    'Usia': 'mean'
}).round(2)
display(summary)